In [1]:
import json
import re
from pathlib import Path
import os
import time
from datetime import datetime
from openai import OpenAI

In [2]:
print("Today:", datetime.now().strftime("%Y-%m-%d"))

Today: 2026-07-12


In [20]:
import math

In [3]:
PROCESSED_DIR  = Path("../data/processed")
FINANCIAL_DIR  = Path("../data/financial_data")
OUTPUT_DIR     = Path("../data/guidance_fulfillment")
OUTPUT_DIR.mkdir(exist_ok=True)


print("Processed transcripts :", PROCESSED_DIR)
print("Financial data        :", FINANCIAL_DIR)
print("Output directory      :", OUTPUT_DIR)

Processed transcripts : ..\data\processed
Financial data        : ..\data\financial_data
Output directory      : ..\data\guidance_fulfillment


In [4]:
from dotenv import load_dotenv
load_dotenv("../.env", override=True)

True

In [5]:
client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [6]:
def extract_guidance_from_transcript(transcript_data):
    """
    Takes processed transcript, extracts all forward guidance claims
    as structured JSON using Groq LLM.
    Returns list of guidance claims.
    """
    guidance_sentences = transcript_data.get("forward_guidance", [])
    if not guidance_sentences:
        return []
        
    symbol = transcript_data["symbol"]
    quarter = transcript_data["quarter"]
    year = transcript_data["year"]

    


    sentences_text = "\n".join(
        f"{i+1}. {s}" for i, s in enumerate(guidance_sentences)
    )

    prompt = f"""You are a financial analyst analysing forward guidance from an earnings call.

Company: {symbol}
Transcript Quarter: Q{quarter} {year}

Below are sentences that may contain forward guidance (management forecasts about future performance).
For each sentence that contains REAL forward guidance (actual numerical or directional forecasts), extract structured data.
Skip sentences that are just disclaimers, boilerplate, or vague statements with no specific forecast metric.

Sentences:
{sentences_text}

Return a JSON array. Each item must have exactly these fields:
[
  {{
    "metric": "exact metric name e.g. Gross Margin, Revenue, EPS, Operating Income, Net Income, CapEx, R&D",
    "value": "exact value or range e.g. 43.5-44.5% or $90B-$92B or high single digits",
    "value_low": null or numeric lower bound e.g. 43.5,
    "value_high": null or numeric upper bound e.g. 44.5,
    "value_unit": "% or B or M or absolute",
    "direction": "increase" or "decrease" or "flat" or "range" or "unknown",
    "target_quarter": null or integer 1-4,
    "target_year": null or integer e.g. 2023,
    "raw_sentence": "original sentence"
  }}
]

Important for target_quarter and target_year:
- "next quarter" from Q{quarter} {year} means Q{quarter % 4 + 1} {year if quarter < 4 else year + 1}
- "full year" or "fiscal year" means all of {year}
- "FY{year+1}" means target_year={year+1}, target_quarter=null
- If no specific time mentioned, assume next quarter

Return ONLY the JSON array, no other text."""
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system",
             "content": "You are a strict financial analyst. Follow the instruction to find out genuine future guidance."
            },
            {"role": "user",
             "content": prompt
            }
        ],
        temperature=0.1
    )
    text = response.choices[0].message.content.strip()
    text = text.replace("```json", "").replace("```", "").strip()

    try:
        result = json.loads(text)
        return result if isinstance(result, list) else []
    except Exception as e:
        print(f"  Extraction error: {e}")
        return []


In [24]:

# Test on AAPL Q2 2022 (guidance for Q3 2022 which has already happened)
test_file = PROCESSED_DIR / "MET_Q2_2024.json"
with open(test_file, encoding="utf-8") as f:
    test_transcript = json.load(f)

print(f"Testing on: {test_transcript['symbol']} Q{test_transcript['quarter']} {test_transcript['year']}")
print(f"Forward guidance sentences: {len(test_transcript['forward_guidance'])}")
print()

guidance_claims = extract_guidance_from_transcript(test_transcript)

print(f"Guidance claims extracted: {len(guidance_claims)}")
print()
for claim in guidance_claims:
    print(f"  Metric: {claim['metric']}")
    print(f"  Value: {claim['value']}")
    print(f"  Value Low/High: {claim['value_low']} / {claim['value_high']}")
    print(f"  Unit: {claim['value_unit']}")
    print(f"  Target: Q{claim['target_quarter']} {claim['target_year']}")
    print(f"  Sentence: {claim['raw_sentence'][:100]}")
    print()

Testing on: MET Q2 2024
Forward guidance sentences: 7

Guidance claims extracted: 5

  Metric: Spreads
  Value: 115-140 basis points
  Value Low/High: 115 / 140
  Unit: basis points
  Target: Q3 2024
  Sentence: We anticipate that spreads will remain between our annual target range of 115 and 140 basis points i

  Metric: Direct Expense Ratio
  Value: 12.3% or below
  Value Low/High: 0 / 12.3
  Unit: %
  Target: QNone 2024
  Sentence: That said, our performance year-to-date positions us well to achieve a full year 2024 direct expense

  Metric: Japan's Solvency Margin Ratio
  Value: approximately 670%
  Value Low/High: None / None
  Unit: %
  Target: Q2 2024
  Sentence: Finally, we expect that Japan's solvency margin ratio to be approximately 670% as of June 30, which 

  Metric: New Money Yields
  Value: above roll-off yields
  Value Low/High: None / None
  Unit: absolute
  Target: Q3 2024
  Sentence: We anticipate that the new money yields will remain above roll-off yields given the 

In [11]:
def get_financial_data_for_quarter(symbol, quarter, year):
    """
    Loads existing financial data file and finds the income statement
    entry for the specific target quarter/year.
    Returns the full income statement row as a dict, or None if not found.
    """
    fin_path = FINANCIAL_DIR / f"{symbol}_financials.json"

    if not fin_path.exists():
        return None

    with open(fin_path, encoding="utf-8") as f:
        fin_data = json.load(f)

    quarterly_income = fin_data.get("income_statement",[])#.get("quarterly_income", [])
    for entry in quarterly_income:
        date = entry.get("fiscalDateEnding", "")
        if not date or len(date) < 7:
            continue
        entry_year    = int(date[:4])
        entry_month   = int(date[5:7])
        entry_quarter = (entry_month - 1) // 3 + 1

        if entry_year == year and entry_quarter == quarter:
            return entry

    return None


# Test: find AAPL Q3 2022 financial data
fin_entry = get_financial_data_for_quarter("AAPL", 3, 2022)
print("AAPL Q3 2022 financial data:")
if fin_entry:
    for k, v in fin_entry.items():
        print(f"  {k}: {v}")
else:
    print("  Not found")

AAPL Q3 2022 financial data:
  fiscalDateEnding: 2022-09-30
  reportedCurrency: USD
  grossProfit: 38095000000
  totalRevenue: 90146000000
  costOfRevenue: 52051000000
  costofGoodsAndServicesSold: 52051000000
  operatingIncome: 24894000000
  sellingGeneralAndAdministrative: 6440000000
  researchAndDevelopment: 6761000000
  operatingExpenses: 13201000000
  investmentIncomeNet: None
  netInterestIncome: -74000000
  interestIncome: 753000000
  interestExpense: 827000000
  nonInterestIncome: None
  otherNonOperatingIncome: 237000000
  depreciation: None
  depreciationAndAmortization: 2865000000
  incomeBeforeTax: 24657000000
  incomeTaxExpense: 3936000000
  interestAndDebtExpense: None
  netIncomeFromContinuingOperations: 20721000000
  comprehensiveIncomeNetOfTax: None
  ebit: 25484000000
  ebitda: 28349000000
  netIncome: 20721000000


In [14]:
def get_earning_history_for_quarter(symbol, quarter, year):
    """
    Loads existing financial data file and finds the income statement
    entry for the specific target quarter/year.
    Returns the full income statement row as a dict, or None if not found.
    """
    fin_path = FINANCIAL_DIR / f"{symbol}_financials.json"

    if not fin_path.exists():
        return None

    with open(fin_path, encoding="utf-8") as f:
        fin_data = json.load(f)

    earning_history = fin_data.get("earnings_history",[])
    for entry in earning_history:
        date = entry.get("fiscalDateEnding", "")
        if not date or len(date) < 7:
            continue
        entry_year    = int(date[:4])
        entry_month   = int(date[5:7])
        entry_quarter = (entry_month - 1) // 3 + 1

        if entry_year == year and entry_quarter == quarter:
            return entry

    return None


# Test: find AAPL Q3 2022 financial data
earning_entry = get_earning_history_for_quarter("AAPL", 3, 2022)
print("AAPL Q3 2022 earning data:")
if earning_entry:
    for k, v in earning_entry.items():
        print(f"  {k}: {v}")
else:
    print("  Not found")

AAPL Q3 2022 earning data:
  fiscalDateEnding: 2022-09-30
  reportedDate: 2022-10-27
  reportedEPS: 1.29
  estimatedEPS: 1.27
  surprise: 0.02
  surprisePercentage: 1.5748
  reportTime: post-market


In [19]:
def compute_metric_from_financials(metric_name, financial_entry, earning_entry):
    """
    Given a metric name (e.g. 'Gross Margin') and a full income statement
    entry dict, use Groq to calculate the actual value of that metric.
    Returns dict with computed_value and unit.
    """
    if not financial_entry and not earning_entry:
        return None

    # Build a clean readable version of the financial data
    fin_text = "\n".join(
        f"  {k}: {v}"
        for k, v in financial_entry.items()
        if v not in (None, "None", "")
    )
    earning_text = "\n".join(
        f"  {k}: {v}"
        for k, v in earning_entry.items()
        if v not in (None, "None", "")
    )


    prompt = f"""You are a financial analyst. 
Given this income statement data for one quarter, calculate the value of: {metric_name}

Income Statement Data:
{fin_text}
Earnings History of this Quarter: 
{earning_text}
Instructions:
For each extracted guidance metric:
1. If the metric exists directly in the income statement,
   return the corresponding value.
2. If the metric exists in earnings history,
   return the corresponding value.
3. If the metric can be derived mathematically from
   available income statement fields or earnings history
   compute it using standard financial formulas.
4. If the metric cannot be computed or found from the
   provided financial data and also from earning history return null.

- All dollar values are in raw numbers (divide by 1B for billions)
Return ONLY this JSON, no other text:
{{
  "metric": "{metric_name}",
  "computed_value": <number or null>,
  "unit": "%" or "B" or "M" or "absolute",
  "explanation": "brief one line explanation of how you calculated it"
}}"""
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system",
             "content": "You are a strict financial analyst. Follow the instruction to calculate the desired metric from given financial data."
            },
            {"role": "user",
             "content": prompt
            }
        ],
        temperature=0.3
    )
    text = response.choices[0].message.content.strip()
    text = text.replace("```json", "").replace("```", "").strip()
    try:
        return json.loads(text)
    except Exception as e:
        print(f"  Compute metric error: {e}")
        return None


# Test: compute Gross Margin from AAPL Q3 2022
result = compute_metric_from_financials("Operating Margin %", fin_entry, earning_entry)
print("Computed Gross Margin for AAPL Q3 2022:")
print(json.dumps(result, indent=2))

Computed Gross Margin for AAPL Q3 2022:
{
  "metric": "Operating Margin %",
  "computed_value": 27.63,
  "unit": "%",
  "explanation": "Calculated as (operatingIncome / totalRevenue) * 100"
}


In [25]:
def score_fulfillment(claim, actual_result):
    """
    Compare extracted guidance claim against actual computed value.
    Returns fulfillment score 0.0 to 1.0.

    Scoring logic:
    - Actual >= guidance low end/guidance high end -> 1.0 (fully beat)
    - Actual < guidance low end -> abs(actual-guidance)/guidance then Score=e^(−5×RelativeError)
    """
    if not actual_result or actual_result.get("computed_value") is None:
        return None

    actual_value = float(actual_result["computed_value"])
    value_low    = claim.get("value_low")
    value_high   = claim.get("value_high")
    unit         = claim.get("value_unit", "%")
    direction    = claim.get("direction", "unknown")

    # Convert revenue/income from raw to billions if needed
    if unit == "B" and actual_value > 1000:
        actual_value = actual_value / 1_000_000_000
    elif unit == "M" and actual_value > 1000:
        actual_value = actual_value / 1_000_000

    # Case 1: specific range given (e.g. 43.5-44.5%)
    if value_low is not None:
        value_low  = float(value_low)
        value_high = float(value_high)

        if actual_value >= value_high or actual_value >= value_low:
            return 1.0  # beat end
        else:
            # below range — score based on how close
            miss_pct = abs(value_low - actual_value) / value_low
            score = math.exp(-5 * miss_pct)
            return score

    # Case 3: only direction given (increase/decrease/flat)
    elif direction == "increase":
        return 0.5
    elif direction == "flat":
        return 0.75
    else:
        return None

In [28]:
# Test scoring
all_results = []
symbol = "MET"
if guidance_claims:
    for claim in guidance_claims:
        metric = claim["metric"]
        target_year = claim["target_year"]
        target_quarter = claim["target_quarter"]
    
        # Income Statement
        financial_entry = get_financial_data_for_quarter(
            symbol,
            target_quarter,
            target_year
        )
    
        # Earnings History
        earnings_entry = get_earning_history_for_quarter(
            symbol,
            target_quarter,
            target_year
        )
    
        actual = compute_metric_from_financials(
            metric,
            financial_entry,
            earnings_entry
        )
    
        score = score_fulfillment(
            claim,
            actual
        )
    
        all_results.append({
            "metric": metric,
            "guidance": claim["value"],
            "actual": actual,
            "score": score
    
        })

In [29]:
print(len(all_results))

5


In [30]:
print(all_results)

[{'metric': 'Spreads', 'guidance': '115-140 basis points', 'actual': {'metric': 'Spreads', 'computed_value': 29.73, 'unit': '%', 'explanation': 'Calculated as (grossProfit / totalRevenue) * 100'}, 'score': 0.024541461041018307}, {'metric': 'Direct Expense Ratio', 'guidance': '12.3% or below', 'actual': None, 'score': None}, {'metric': "Japan's Solvency Margin Ratio", 'guidance': 'approximately 670%', 'actual': {'metric': "Japan's Solvency Margin Ratio", 'computed_value': None, 'unit': 'absolute', 'explanation': 'Cannot be computed from provided income statement and earnings history data as it requires specific balance sheet and regulatory data'}, 'score': None}, {'metric': 'New Money Yields', 'guidance': 'above roll-off yields', 'actual': {'metric': 'New Money Yields', 'computed_value': None, 'unit': 'absolute', 'explanation': 'Cannot be computed from provided income statement and earnings history data'}, 'score': None}, {'metric': 'VII Returns', 'guidance': 'continue to improve', 'act

In [7]:
METRIC_REGISTRY = {

    "Revenue": {
        "source": "income_statement",
        "field": "totalRevenue",
        "type": "direct"
    },

    "Gross Profit": {
        "source": "income_statement",
        "field": "grossProfit",
        "type": "direct"
    },

    "Operating Income": {
        "source": "income_statement",
        "field": "operatingIncome",
        "type": "direct"
    },

    "Operating Margin": {
        "source": "income_statement",
        "type": "derived"
    },

    "Gross Margin": {
        "source": "income_statement",
        "type": "derived"
    },

    "Net Income": {
        "source": "income_statement",
        "field": "netIncome",
        "type": "direct"
    },

    "OpEx": {
        "source": "income_statement",
        "field": "operatingExpenses",
        "type": "direct"
    },

    "Operating Expenses": {
        "source": "income_statement",
        "field": "operatingExpenses",
        "type": "direct"
    },

    "R&D": {
        "source": "income_statement",
        "field": "researchAndDevelopment",
        "type": "direct"
    },

    "CapEx": {
        "source": "income_statement",
        "field": None,
        "type": "unsupported"
    },

    "EPS": {
        "source": "earnings_history",
        "field": "reportedEPS",
        "type": "direct"
    }
}